# 02 — GARCH(1,1): fit, forecast, and an honest verdict

**Companion:** [docs/concept-guide.md](../docs/concept-guide.md) §§4–5 · **Timeboxes:** U1–U4 in [ROADMAP.md](../ROADMAP.md)

**Goal:** fit GARCH(1,1) on the first 70% of the sample, produce one-day-ahead variance forecasts over the last 30% with parameters frozen at their train estimates, and score them against two naive benchmarks under QLIKE (primary) and MSE (secondary).

**Reminder: an undramatic result is the expected, publishable finding.** GARCH should beat the unconditional benchmark clearly and naive persistence modestly, if at all (guide §5.2). Your job is to measure that cleanly, not to make the model look good. Same four rules as notebook 01.

In [ ]:
# GIVEN — notebook 02 shares no kernel state with notebook 01: everything reloads here.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import plotly.graph_objects as go          # go.Figure / go.Scatter / go.Bar
from plotly.subplots import make_subplots
from arch import arch_model

# Static PNG for the README:  fig.write_image("figures/figN.png")   (kaleido is installed)

from garch_lab.data import load_prices
from garch_lab.returns import log_returns            # yours since Saturday
from garch_lab.volatility import rolling_vol         # yours since Saturday
from garch_lab.evaluation import (                   # yours this morning (box U1)
    chronological_split, mse, qlike, rw_variance_forecast,
)

px = load_prices("^GSPC", start="2005-01-01")        # cache hit - no network needed
r = log_returns(px)

## §5.1 first: the split — guide §5.1

Evaluation design comes *before* the model. Deciding it after seeing forecasts is how look-ahead seeps in.

- Q1. Why is a random train/test split not merely suboptimal but *invalid* for serially dependent data? What exactly leaks?
- Q2. From here on, every quantity you compute carries a mental timestamp of when it became knowable. What does that rule say about anything "full-sample"?

**Task** — implement all four functions in [src/garch_lab/evaluation.py](../src/garch_lab/evaluation.py). They are pure functions — no model needed; this is the Sunday warm-up.

**DONE when:** `pytest` is fully green. Then split `r` 70/30 chronologically and record the split date.

<details><summary>Hint — the only one</summary>

Each contract docstring names the property; each failing test names the violation. For `rw_variance_forecast`, count the NaNs before reaching for anything clever — the warm-up length IS the specification of the shift.

</details>

In [ ]:
# TODO(you): make the full suite green, then:
#   train, test = chronological_split(r, 0.7)
#   split_date = ...

### ✓ Checkpoint 1

- len(test) ≈ 1,600 ± 50; train ends and test begins on adjacent trading days.
- Full suite green — the natural place for a commit.

➜ **Look-ahead audit, one sentence:** name every full-sample statistic you have already produced this weekend (they all live in notebook 01, computed on what is now train+test), and say why that is acceptable for *description* but would not be for *modelling*.

_Your sentence:_

## §4.1 Fitting GARCH(1,1) — guide §4.1

**Definition — GARCH(1,1).** Returns are $r_t = \sigma_t z_t$ with $z_t \overset{iid}{\sim} (0,1)$, and the conditional variance follows

$$
\sigma_t^2 = \omega + \alpha\, \varepsilon_{t-1}^2 + \beta\, \sigma_{t-1}^2, \qquad \varepsilon_{t-1} = r_{t-1},
$$

with $\omega > 0$, $\alpha, \beta \ge 0$, and covariance-stationarity requiring $\alpha + \beta < 1$, in which case the unconditional variance is $\sigma^2 = \omega / (1 - \alpha - \beta)$.

- Q1. Before fitting: map ω, α, β to *baseline / reaction to news / memory*.
- Q2. From the ACF decay in your fig1, write down a guess for α+β to one decimal. Check yourself after the fit.
- Q3. Why does the optimizer want returns ×100 — what is badly conditioned about a likelihood over variances of order 10⁻⁴?

**Task:** fit GARCH(1,1) on the **train window only** — once with Gaussian innovations, once with Student-t.

**DONE when:** both fits converge and the checkpoint holds.

<details><summary>Hint 1 — API shape</summary>

`arch_model(100*r, vol="GARCH", p=1, q=1, dist=...)`. Fitting on unscaled returns triggers a `DataScaleWarning` — heed it (guide §4.3).

</details>

<details><summary>Hint 2 — fitting on train only, without slicing</summary>

Construct the model on the FULL series but pass `last_obs=` to `.fit()` — estimation then uses only data before the cutoff, and the forecast step can later roll through the test window with frozen parameters. Is `last_obs` itself included in estimation? Do not trust the docs or me: check "No. Observations" in the fit summary against your train length.

</details>

<details><summary>Hint 3 — believing the output</summary>

</details>

Find the attribute on the results object that equals 0 when the optimizer converged, and assert it before reading anything else. Then parameter sanity: ω > 0; α, β ≥ 0; α + β < 1.

In [ ]:
# TODO(you): fit GARCH(1,1)-N and GARCH(1,1)-t on the train window only;
#            print both summaries; assert convergence

### ✓ Checkpoint 2

- α ∈ [0.05, 0.20], β ∈ [0.78, 0.94], α+β ∈ [0.95, 0.999].
- Shock half-life ln 2 / |ln(α+β)| ≈ 15–100 days (α+β = 0.98 → ≈ 34).
- Implied unconditional vol √(252 · ω/(1−α−β)) ≈ 15–22% annualized — compare against your train-sample vol.
- Student-t ν̂ ∈ [4, 10], and **larger than notebook 01's unconditional ν̂**.

➜ **Two sentences:** (1) why is the *conditional* ν̂ larger — i.e. why does conditioning on σ_t absorb part of the tails? (Guide §4.2, the mixture argument.) (2) Why would comparing this log-likelihood with one from a fit on unscaled returns be meaningless?

_Your sentences:_

## §4.2 In-sample sanity — the filter view

GARCH is your boxcar upgraded to a learned exponential filter. Look at them side by side before trusting the model out of sample.

**Task:** overlay the fitted conditional volatility (annualized) with your 21-day rolling vol over the full period; mark the train/test boundary with a vertical line; export `figures/fig3_rolling_vs_garch.png` (supersedes yesterday's draft — this version takes the README slot).

<details><summary>Hint — units</summary>

`res.conditional_volatility` is daily, in percent (you fit on 100·r). Your `rolling_vol` on raw `r` is daily, in raw units. Convert BOTH to annualized percent before plotting, or the plot will shout the ×100 / ×15.9 tells at you. The train/test divider is one line: `fig.add_vline(x=split_date, line_dash="dash")`.

</details>

In [ ]:
# TODO(you): overlay annualized conditional vol vs 21d rolling vol, train/test divider;
#            export figures/fig3_rolling_vs_garch.png

### ✓ Checkpoint 3

- Same regimes in both series, but GARCH is spikier and decays faster after shocks — exponential memory vs boxcar.

➜ **One sentence:** what can the fitted filter do that the boxcar cannot?

_Your sentence:_

## §5.1 One-step out-of-sample forecasts — the alignment discipline

- Q. The forecast made on day t targets day t+1. In a series indexed by *target date*, which row must that forecast occupy — and what one-line `assert` proves you did not cheat?

**Task:** one-step-ahead GARCH variance forecasts over the whole test window (both fits); the proxy p = (100·r_test)²; and both benchmarks — random walk via YOUR `rw_variance_forecast`, unconditional = the train mean of (100·r)² broadcast over the test index.

**DONE when:** every series lives on the test index, in %²-per-day units, and the perturbation check below passes.

<details><summary>Hint 1 — the forecast call</summary>

`res.forecast(horizon=1, start=..., reindex=False)`, column `h.1`. No refit loop needed: the parameters stay frozen at their train estimates while the σ² recursion rolls through the observed test returns.

</details>

<details><summary>Hint 2 — alignment</summary>

In the forecast frame, the value in row t was made WITH information through t and is a forecast OF t+1. One pandas one-liner moves each value to the date it targets. Finish with `assert h.index.equals(p.index)`.

</details>

<details><summary>Hint 3 — units</summary>

Feed the SAME percent-scaled returns (100·r) to every forecast, benchmarks included. Daily variance in %², everywhere. No square roots, no 252, until a plot axis needs them.

</details>

In [ ]:
# TODO(you): h_garch_n, h_garch_t, h_rw, h_unc, p
#            -> all on the test index, all in %^2 per day

### ✓ Checkpoint 4 — prove there is no look-ahead

- First forecast's target = the first test day; the index `assert` passes.
- **Perturbation self-check (3 lines — actually do it):** multiply one mid-test return by 10, rebuild the GARCH forecast series the same way, and confirm the forecast *for* that date is unchanged while later ones move. Same property your `rw_variance_forecast` test enforced this morning.
- Rough level check: mean(h_GARCH) within ~×1.3 of mean(p) over the test window.

➜ **One sentence:** where exactly would the cheat have hidden if you had skipped the shift?

_Your sentence:_

## §5.2–5.3 Scoring — guide §§5.2–5.3

**Definition — QLIKE loss.** For a variance proxy $p_t$ and forecast $h_t$, $\ \mathrm{QLIKE}_t = \dfrac{p_t}{h_t} - \ln\dfrac{p_t}{h_t} - 1$.

**Definition — MSE loss.** $\mathrm{MSE}_t = (p_t - h_t)^2$.

Both are averaged over the test window to score a forecast; QLIKE is scale-invariant and penalizes under-prediction more heavily (Patton 2011).

**Task:** the loss table — four forecasts × two losses. FIRST count zero-return days in the test window: your `qlike` refuses p ≤ 0 by contract, so decide (drop with disclosure, or clip to a tiny ε) and write the count down for the README.

- Q1. Which of your two losses will a single March-2020 day dominate, and through what feature of the loss?

- Q2. By hand: QLIKE at h = p/2 vs h = 2p. Which is bigger — and why is that the right sign for a risk manager?- Q3. The proxy's noise is ≈ 141% of its signal (guide §3.3). One line: why do MSE and QLIKE survive that while MAE, MSE-on-σ, and percentage errors can misrank models? (Patton 2011.)

In [ ]:
# TODO(you): zero-return-day count + your decision;
#            loss table: {GARCH-t, GARCH-N, random walk, unconditional} x {QLIKE, MSE}

### ✓ Checkpoint 5

- QLIKE ordering: GARCH < random walk < unconditional.
- GARCH beats unconditional decisively (tens of %); beats the random walk modestly (0–10%) or not at all — **at a one-day horizon that is the expected, honest result** (guide §5.2).
- MSE may rank GARCH vs random walk either way. If it disagrees with QLIKE, that is a finding about MSE's crisis-day fragility, not about the models.

➜ **One sentence per row** of the table, in plain language.

_Your sentences:_

## Wrap — the sober conclusion

Write one paragraph, numbers included, answering the README's title question. This paragraph IS your README TL;DR — write it once, here, in your own voice.

_Your paragraph:_

**Next steps** (each tied to something you measured — this list goes in the README):

- Leverage effect (nb01 Checkpoint 5) unmodelled — GARCH(1,1) is sign-blind → GJR-GARCH / EGARCH.
- ACF of |r| decays slower than any exponential (fig1) → HAR-RV on true realized volatility.
- Single split, frozen parameters → walk-forward re-estimation.
- The QLIKE gap carries no significance statement → Diebold–Mariano test with a HAC standard error.

**Housekeeping before you push** (ROADMAP box U4): `pip freeze > requirements.txt` · your name in LICENSE · delete the README skeleton comments · Restart & Run All on both notebooks.